In [20]:
import pandas as pd

In [21]:
import numpy as np

In [22]:
import matplotlib.pyplot as plt

In [23]:
from sklearn.decomposition import TruncatedSVD

In [24]:
df = pd.read_csv("/Users/senuja/Jupyter Notebook/Video_Game_Sales_Predictor/data/cleaned/5. Feature Selection/Feature Selection(1-hot).csv")

In [25]:
df.head()

,Game Title,Critic Score,Sales,North American Sales,Japanese Sales,EU Sales,Other Sales,Release Year,Genre_Action-Adventure,Genre_Adventure,...,Developer_Vicarious Visions,Developer_Visual Concepts,Decade,NA Share,EU Share,JP Share,Other Share,Is Recent,Game Quality,Game Quality Encoded
0,Grand Theft Auto V,0.933333,1.000000,0.652664,0.464789,1.000000,1.000000,2013,False,False,...,False,False,2010,0.652664,1.000000,0.464789,1.000000,0,High,2
1,Grand Theft Auto V,0.966667,0.953740,0.620902,0.281690,0.985787,0.967949,2014,False,False,...,False,False,2010,0.651018,1.033601,0.295353,1.014898,0,High,2
2,Grand Theft Auto: Vice City,0.955556,0.794291,0.861680,0.220657,0.557360,0.570513,2002,False,False,...,False,False,2000,1.084842,0.701708,0.277804,0.718266,0,High,2
3,Grand Theft Auto V,0.677988,0.780512,0.928279,0.028169,0.541117,0.455128,2013,False,False,...,False,False,2010,1.189320,0.693285,0.036090,0.583115,0,Medium,1
4,Call of Duty: Black Ops 3,0.788889,0.742126,0.633197,0.192488,0.614213,0.782051,2015,False,False,...,False,False,2010,0.853220,0.827640,0.259374,1.053799,1,High,2


In [26]:
df.dtypes # check data types

Game Title               object
Critic Score            float64
Sales                   float64
North American Sales    float64
Japanese Sales          float64
                         ...   
JP Share                float64
Other Share             float64
Is Recent                 int64
Game Quality             object
Game Quality Encoded      int64
Length: 133, dtype: object

In [27]:
# 2) Build numeric matrix (drop target), clean, and make floats
X = df.drop(columns=["Sales"]).select_dtypes(include=["number", "bool"]).copy()
X = X.replace([np.inf, -np.inf], np.nan).fillna(0)
X = X.astype(float)

In [28]:
# 3) Find K to hit ~90% variance
max_k = min(50, X.shape[1] - 1) if X.shape[1] > 1 else 1
svd_probe = TruncatedSVD(n_components=max_k, random_state=42)
svd_probe.fit(X)
cumvar = svd_probe.explained_variance_ratio_.cumsum()

# Auto-pick K
K = int(np.searchsorted(cumvar, 0.90) + 1) if X.shape[1] > 1 else 1
K = max(2, min(K, max_k))
print("Chosen components K =", K, "| Cumulative variance ≈", round(cumvar[K-1], 4))

Chosen components K = 2 | Cumulative variance ≈ 0.922


In [29]:
# 4) Final SVD transform
svd = TruncatedSVD(n_components=K, random_state=42)
Z = svd.fit_transform(X)
comp_cols = [f"SVD_{i+1}" for i in range(K)]

In [30]:
# 5) OVERWRITE df with reduced components + Sales
df = pd.concat(
    [df[["Sales"]].reset_index(drop=True),
     pd.DataFrame(Z, columns=comp_cols)],
    axis=1
)

print("AFTER  (rows, cols):", df.shape)
display(df.head())

AFTER  (rows, cols): (17567, 3)


,Sales,SVD_1,SVD_2
0,1.000000,2844.691573,-2.166189
1,0.953740,2845.399379,-2.340130
2,0.794291,2829.841407,-2.355597
3,0.780512,2844.690798,-2.614922
4,0.742126,2846.107263,-2.399483


In [34]:
df.head()

# SVD_1 - a weighted mix of many original features (genre, console, publisher, critic score, etc.) 
        # chosen to capture the largest possible variance

#SVD_2 - another mix of features, capturing the second-most variance

,Sales,SVD_1,SVD_2
0,1.000000,2844.691573,-2.166189
1,0.953740,2845.399379,-2.340130
2,0.794291,2829.841407,-2.355597
3,0.780512,2844.690798,-2.614922
4,0.742126,2846.107263,-2.399483


In [35]:
df.to_csv("/Users/senuja/Jupyter Notebook/Video_Game_Sales_Predictor/data/cleaned/6. Dimensionality Reduction/Dimensionality Reduction.csv", index=False)